## Initialization

### Imports

In [ ]:
# Importing needed code

import re
import json
from collections import defaultdict
from functools import reduce
from typing import (
    Callable,
    TypeVar,
    Union,
    # Any,
    Literal
)
from datetime import datetime, timezone, timedelta
from math import sqrt, log, ceil
import shutil
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
from mpl_toolkits.axes_grid1 import make_axes_locatable
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from scipy.interpolate import make_interp_spline
from scipy.stats import linregress
from scipy.signal import savgol_filter, find_peaks, peak_prominences, peak_widths
from pint import Quantity
import bottleneck

from data_processing import processing as proc
from data_processing import loading as load
from data_processing import types as proc_types
from data_processing import helpers
from data_processing.paths import (
    get_report_root, get_exp_root, get_reactor_data_root)
from data_processing.dataframe_validation import (
    DetectorDataframeColumn,
    BinningDataframeColumn,
    # NonReactorDataframeColumn,
    # SliceFitDataframeColumn,
    EnergyColumn,
    get_df_col
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
from data_processing.reporting.plotting import plot_scatter, plot_classification
from data_processing.processing.neutron_window_strategy.strategy_factory \
    import NeutronStrategyFactory
from data_processing.processing.neutron_window_strategy.abstract_strategy \
    import AbstractNeutronStrategy
# from data_processing.helpers import (
#     # get_input_with_default,
#     # get_input_required,
#     # input_experiment_ids,
#     stop,
#     get_midpoints_from_bins
# )


### Functions

In [ ]:
CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]

In [ ]:
def bin_non_neutron_data(df, time_bins, data_col, selected_cols):
    start_time = time_bins[0]
    df = get_time_cut(df, 'Time', time_bins)

    binned_df = df.groupby("Time Bin", as_index=False)[data_col] \
        .agg(['mean', 'std']) \
        .copy()
    binned_df.columns = selected_cols
    binned_df['Bin midpoint'] = binned_df.index.to_series() \
        .apply(lambda x: x.mid)
    binned_df = bin_midpoint_time_to_seconds(binned_df, start_time)

    return binned_df

In [ ]:
def bin_midpoint_time_to_seconds(df, start_time):
    bin_mid_col = df[BinningDataframeColumn.BIN_MIDPOINT.value]
    bin_time_col_name = BinningDataframeColumn.BIN_TIME.value
    zeroed_midpoint = pd.to_datetime(bin_mid_col) - start_time
    df[bin_time_col_name] = zeroed_midpoint.dt.total_seconds()
    return df

In [ ]:
def get_time_cut(df, time_tag_col, time_bins):
    timetag_cut = pd.cut(df[time_tag_col], bins=time_bins)
    df[BinningDataframeColumn.TIME_BIN.value] = timetag_cut
    return df

In [ ]:
# fns ask questions, then generate strategy using factory

CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]


def get_nasa_loading_settings(
    calib_key: CalibrationKey
) -> str:
    left_border_type = helpers.get_input_with_default(
        """\
Which left border calculation do you want to use?
1: original left border (0.1966 MeVee)
2: newer left border (~0.1866 MeVee)
3: CAEN lower limit (0.050 MeVee) (default)
Press Enter for default
""",
        3,
        int
    )
    border_key: NasaBorderKey = (
        ExperimentDataKey.NASA_BORDERS if left_border_type == 1 
        else ExperimentDataKey.NASA_BORDERS_RECALC
    )
    file_name_prefix = f"{calib_key.value}_{border_key.value}"
    return file_name_prefix


def get_n_distro_loading_settings(
    calib_key: CalibrationKey
) -> str:
    file_name_prefix = f"{calib_key.value}_{ExperimentDataKey.N_WINDOW_BORDERS.value}"
    return file_name_prefix


def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> proc_types.NasaGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = helpers.get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = helpers.get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = helpers.get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee)
2: newer (~0.1866 MeVee)
3: detector lower limit (0.050 MeVee) (default)
or press Enter for default
""",
            3,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = (
                ExperimentDataKey.NASA_BORDERS 
                if existing_left_border_version_input == 1 
                else ExperimentDataKey.NASA_BORDERS_RECALC
            )
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = load.get_neutron_window_paths(
                file_name_prefix=file_name_prefix)
            left_border, _ = load.load_side_borders(
                side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        elif existing_left_border_version_input == 3:
            lower_energy_bound = 0.05
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = helpers.get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.050)
""",
            0.050,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def get_n_distro_generation_settings(
) -> proc_types.NeutronDistributionGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (3)
""",
        3,
        float
    )
    settings = proc_types.NeutronDistributionGenerationSettings(
        sigma=sigma
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: proc.NeutronStrategyFactory,
    window_type: proc_types.WindowType,
    loading: bool,
    settings: proc_types.NeutronWindowSettings
) -> Callable[[], AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data


In [ ]:
# def relative_rmse(x: pd.Series | float, x_err: pd.Series | float, y: pd.Series | float, y_err: pd.Series | float) -> pd.Series | float:
def relative_rmse(values: list[tuple[pd.Series | float, pd.Series | float]]) -> pd.Series | float:
    rel_sq_values = [relative_square_error(x, x_err) for x, x_err in values]
    # rel_sq_x = relative_square_error(x, x_err)
    # rel_sq_y = relative_square_error(y, y_err)
    # rel_sq_sum = rel_sq_x + rel_sq_y
    rel_sq_sum = sum(rel_sq_values)
    if isinstance(rel_sq_sum, pd.Series):
        return rel_sq_sum.pow(1./2)
    else:
        return rel_sq_sum ** (1./2)


def relative_square_error(x: pd.Series | float, x_err: pd.Series | float) -> pd.Series | float:
    # divide x_err by x
    # square it
    # return
    rel_err = x_err / x
    if isinstance(rel_err, pd.Series):
        return rel_err.pow(2).fillna(0)
    else:
        return rel_err ** 2

In [ ]:
def correct_raw_signals(
    raw_signals_df: pd.DataFrame,
    baseline_idx_range: int = 40,
    baseline_offset: float = 0,
    max_adc: int = 16367,
    use_max_adc: bool = False
) -> pd.DataFrame:
    offset = int(baseline_offset * max_adc)
    signals_np = raw_signals_df.to_numpy()
    
    if use_max_adc:
        baselines = max_adc
    else:
        baselines = signals_np[
            :, :baseline_idx_range
        ].mean(axis=1).reshape(-1, 1)
    
    signals_np = -signals_np + baselines + offset
    corrected_signals = pd.DataFrame(
        signals_np,
        index=raw_signals_df.index,
        columns=raw_signals_df.columns
    )
    return corrected_signals

In [ ]:
def get_psd_adc_histogram(
    df: pd.DataFrame,
    adc_col: str = DetectorDataframeColumn.ENERGY.value,
    psd_col: str = DetectorDataframeColumn.PSD.value,
    adc_width: float = 420,
    adc_bins: np.ndarray | None = None,
    psd_bin_count: int = 100,
    psd_min: float = 0.0,
    psd_max: float = 0.5
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    x = df[adc_col]
    y = df[psd_col]

    within_psd = y.between(psd_min, psd_max)
    x = x[within_psd == True].copy()
    y = y[within_psd == True].copy()

    if adc_bins is not None:
        x_bins = adc_bins
    else:
        x_bins: np.ndarray = np.linspace(
            0, x.max(), int(x.max() / adc_width) + 1
        )
    print(f"Energy width = {x_bins[1]-x_bins[0]} ADC")
    y_bins: np.ndarray = np.linspace(psd_min, psd_max, psd_bin_count + 1)

    Z, xe, ye = np.histogram2d(x, y, bins=[x_bins, y_bins])
    return Z, xe, ye

In [ ]:
def integrate_signals(y: np.array, x: Union[np.array, None] = None) -> float:
    if x is None:
        _, x = np.mgrid[:y.shape[0], :y.shape[1]]
    # TODO test that x and y are 2D
    if len(y.shape) != 2:
        raise ValueError("y array must be 2 dimensional")
    # if x.shape != y.shape:
    #     raise ValueError("x and y must have the same shape")

    left_ys = y[:, :-1]
    right_ys = y[:, 1:]
    left_xs = x[..., :-1]
    right_xs = x[..., 1:]

    # trapezoidal integration
    # area between points has shape of trapezoid
    # trap. area = rectangle where height is mean of parallel edge heights
    trap_mean_heights = (left_ys + right_ys) / 2
    trap_widths = right_xs - left_xs
    trap_areas = trap_mean_heights * trap_widths
    integral = np.sum(trap_areas, axis=1)
    return integral

In [ ]:
def moving_average(arr, n=5):
    ret = np.cumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((n-1,))
    prefix[:] = np.nan
    return np.concatenate((prefix, mov_avg))


def moving_average_centered(arr, n=5):
    if n % 2 != 1:
        raise ValueError("Centered moving average needs odd window size")
    prefix_count = (n-1)//2
    ret = np.nancumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((prefix_count,))
    suffix = np.empty((prefix_count,))
    prefix[:] = np.nan
    suffix[:] = np.nan
    return np.concatenate((prefix, mov_avg, suffix))


def simple_deriv(arr):
    hi = arr[2:]
    lo = arr[:-2]
    delta = hi - lo
    prefix = np.empty((1,))
    suffix = np.empty((1,))
    prefix[:] = np.nan
    suffix[:] = np.nan
    return np.concatenate((prefix, delta, suffix))

In [ ]:
def integrate_pulses(pulse_data: np.array, t_start: int, t_end: int):
    left_vals = pulse_data[:, t_start:t_end]
    right_vals = pulse_data[:, t_start+1:t_end+1]
    # left_vals = pulse_series.loc[t_start:t_end]
    # right_vals = pulse_series.loc[t_start+1:t_end+1]

    # print(left_vals.shape, right_vals.shape)
    # print(left_vals)
    # print(right_vals)
    midpoints = (left_vals + right_vals) / 2
    # print(midpoints)
    column_areas = midpoints * 2  # 2 ns between data points
    # print(column_areas)
    areas = column_areas.sum(axis=1)
    return areas


def integrate_pulse_gates(
    pulses: pd.DataFrame, t1: int, t2: int, t3: int, baseline_count: int = 40
) -> pd.DataFrame:
    baseline_adjust = 0
    # baseline_window = 25

    if not pd.api.types.is_numeric_dtype(pulses.values):
        raise ValueError("DataFrame values must all be numeric type")
    if pulses.shape[1] != 200:
        raise ValueError("DataFrame rows must be 200 samples long")
    if not all([0 <= x <= 398 for x in [t1, t2, t3]]):
        raise ValueError("Times must all be between 0 and 398 (inclusive)")
    if not (t2 > t1):
        raise ValueError("t2 must be greater than t1")
    if not (t3 > t2):
        raise ValueError("t3 must be greater than t2")

    idx_1 = ceil(t1 / 2)
    idx_2 = ceil(t2 / 2)
    idx_3 = ceil(t3 / 2)

    pulses_np = pulses.to_numpy()
    # baselines = np.trunc(
    #     pulses_np[:, :baseline_count].mean(axis=1).reshape(-1, 1)
    # )
    # baselines = baselines + baseline_adjust
    # long_slices = pulses_np[:, idx_1:idx_3]
    # short_slices = pulses_np[:, idx_1:idx_2]
    # peak_slices = pulses_np[:, 30:60]

    # q_long = (baselines - long_slices).sum(axis=1)
    # q_short = (baselines - short_slices).sum(axis=1)
    q_long = integrate_pulses(pulses_np, idx_1, idx_3)
    q_short = integrate_pulses(pulses_np, idx_1, idx_2)
    # peak_heights = (baselines - peak_slices).max(axis=1)

    psd_df = pd.DataFrame(
        {"Q_LONG": q_long, "Q_SHORT": q_short},
        index=pulses.index
    )
    return psd_df

In [ ]:
def two_point_inv_lerp(y: float, p1: tuple[float, float], p2: tuple[float, float]) -> float:
    deltas = tuple([n2 - n1 for n1, n2 in zip(p1, p2)])
    m = deltas[1] / deltas[0]
    x1, y1 = p1
    if m == 0:
        return x1
    x = (y - y1) / m + x1
    return x


def calculate_q_fom_critical(
    fit_df: pd.DataFrame,
    # q_limits: tuple[float, float] | None = None
    # q_limit_hi: float | None = None
) -> tuple[float | None, float | None]:
    fom_crit = 1.27
    # if q_limits is None:
    #     q_limits = (2500, 60000)  # x axis area with clean FOM curve
    # if q_limit_hi is None:
    #     q_limit_hi = 60000
    fom_data = fit_df["fom"]
    slice_energy_min = fit_df["slice_energy_min"]
    slice_energy_max = fit_df["slice_energy_max"]
    slice_energy_mid = (slice_energy_min + slice_energy_max) / 2
    fom_x = slice_energy_mid.values
    fom_y = fom_data.values
    delta_y = fom_y[2:] - fom_y[:-2]
    stable_end_idx = np.argmax(fom_x >= 20000)

    # use moving average of delta_y to find stable region (delta_y <= threshold)
    # find first cross in stable region
    window = 5
    # bottleneck window functions use look-behind windows and fill missing with nan
    # so first window-1 values are always nan; we need to convert to look-ahead
    # 
    delta_y_mov_max = bottleneck.move_max(
        delta_y[:stable_end_idx+window-1], window
    )[window-1:]
    delta_y_mov_min = bottleneck.move_min(
        delta_y[:stable_end_idx+window-1], window
    )[window-1:]
    stable_max_delta = delta_y_mov_max < 0.25
    stable_min_delta = delta_y_mov_min > -0.25
    stable_delta = (stable_max_delta & stable_min_delta)
    if not stable_delta.any():
        return None, None
    stable_start_idx = np.argmax(stable_delta)
    stable_start_x = fom_x[stable_start_idx]
    cross_search_slice = fom_y[stable_start_idx:stable_end_idx]
    if not (cross_search_slice >= fom_crit).any():
        return None, stable_start_x
    # argmax gets i in slice, need to add slice start to get i in original list
    fom_critical_idx = np.argmax(cross_search_slice >= fom_crit) + stable_start_idx
    
    # possible_crosses = np.where((fom_y[1:] >= 1.27) & (fom_y[:-1] <= 1.27))[0]
    # # dd_sums = []
    # fom_critical_idx = None
    # for possible_cross in possible_crosses:
    #     pass  # STUB
    # slice_lo = possible_cross - 3 if possible_cross - 3 >= 0 else 0
    # slice_hi = possible_cross + 2
    # deltas = delta_y[slice_lo:slice_hi]
    # delta_deltas = deltas[1:] - deltas[:-1]
    # dd_sum = abs(delta_deltas).sum()
    # dd_sums.append(dd_sum)
    
    # idx_best_cross = np.argmin(np.nan_to_num(dd_sums, nan=np.inf))
    # fom_critical_idx = possible_crosses[idx_best_cross] + 1
    if fom_critical_idx is None:
        q_fom_critical = None
    elif fom_critical_idx > 0:
        # x_crit_bounds = tuple([fom_x[fom_critical_idx+x] for x in [-1, 0]])
        # y_crit_bounds = tuple([fom_y[fom_critical_idx+x] for x in [-1, 0]])
        p1 = fom_x[fom_critical_idx-1], fom_y[fom_critical_idx-1]
        p2 = fom_x[fom_critical_idx], fom_y[fom_critical_idx]
        if p1[1] > fom_crit:  # we can't make lerp extrapolate!
            q_fom_critical = None
        else:
            q_fom_critical = two_point_inv_lerp(fom_crit, p1, p2)
    else:
        q_fom_critical = fom_x[fom_critical_idx]
    return q_fom_critical, stable_start_x

In [ ]:
def calculate_q_fom_critical_from_pulses(
    times: np.ndarray, pulses: pd.DataFrame
) -> tuple[float | None, float | None]:
    t1, t2, t3, *_ = times.flatten()
    psd_df = integrate_pulses(pulses, t1, t2, t3)
    psd_df = calculate_psd(psd_df)
    histogram, energy_bin_edges, psd_bin_edges = get_psd_energy_histogram(psd_df)
    fit_df, _ = scan_histogram_slices(histogram, energy_bin_edges, psd_bin_edges)
    q_fom_critical = calculate_q_fom_critical(fit_df)
    return q_fom_critical


def generate_search_grid(
    t1_limits: tuple[float, float],
    t2_limits: tuple[float, float],
    t3_limits: tuple[float, float],
    counts: int | tuple[int, int, int],
) -> np.ndarray:
    if isinstance(counts, tuple):
        t1_counts, t2_counts, t3_counts = counts
    else:
        t1_counts, t2_counts, t3_counts = counts, counts, counts
    t1_range = np.linspace(*t1_limits, num=t1_counts)
    t2_range = np.linspace(*t2_limits, num=t2_counts)
    t3_range = np.linspace(*t3_limits, num=t3_counts)
    t_grid = np.meshgrid(t1_range, t2_range, t3_range)
    t_grid = tuple([np.ravel(grid_element) for grid_element in t_grid])
    t_grid_stacked = np.vstack(t_grid)  # shape (3, n)
    return t_grid_stacked


T = TypeVar("T")


def search_grid(
    grid_search_fn: Callable[[np.ndarray, pd.DataFrame], T],
    t1_limits: tuple[float, float],
    t2_limits: tuple[float, float],
    t3_limits: tuple[float, float],
    counts: int | tuple[int, int, int],
    pulses: pd.DataFrame,
    cores: int = 4,
    use_chunks: bool = False,
) -> tuple[np.ndarray, list[T]]:
    t_grid_stacked = generate_search_grid(
        t1_limits, t2_limits, t3_limits, counts
    )

    pool_size = max(
        2 * cores, 4
    )
    # based on https://jupyter-tutorial.readthedocs.io/en/stable
    # /performance/multiprocessing.html
    if use_chunks:
        chunksize, extra = divmod(len(t_grid_stacked.shape[1]), pool_size * 4)
        if extra > 0:
            chunksize += 1
    else:
        chunksize = "auto"

    parallelizer = Parallel(
        n_jobs=pool_size, batch_size=chunksize, max_nbytes=1e6, verbose=10
    )
    loop_result = parallelizer(
        delayed(grid_search_fn)(timesarray, pulses)
        for timesarray in t_grid_stacked.T
    )
    return t_grid_stacked, loop_result


def grid_search_fom(
    t1_limits: tuple[float, float],
    t2_limits: tuple[float, float],
    t3_limits: tuple[float, float],
    counts: int | tuple[int, int, int],
    pulses: pd.DataFrame,
    cores: int = 4,
    use_chunks: bool = False,
) -> np.ndarray:
    t_grid_stacked, fom_values = search_grid(
        calculate_q_fom_critical_from_pulses,
        t1_limits,
        t2_limits,
        t3_limits,
        counts,
        pulses,
        cores,
        use_chunks
    )
    
    threshold_energies, threshold_search_starts = list(zip(*fom_values))
    threshold_energies = np.array(threshold_energies, dtype=float)
    threshold_search_starts = np.array(threshold_search_starts, dtype=float)

    return np.vstack(
        [t_grid_stacked, threshold_energies, threshold_search_starts],
        dtype=float
    ).T  # shape (n, 5)

In [ ]:
def calculate_plot_grid_dimensions(n: int, max_cols: int = 4) -> tuple[int, int]:
    ncols = min(n, max_cols)
    nrows = ceil(n / ncols)
    return (nrows, ncols)

In [ ]:
def display_plot_grid(
    grid_plot_fn: Callable[[mpl.axes.Axes, T], None],
    grid_plot_data: list[T],
    grid_count: int,
    max_cols: int
) -> tuple[mpl.figure.Figure, np.ndarray[mpl.axes.Axes]]:
    nrows, ncols = calculate_plot_grid_dimensions(grid_count, max_cols=max_cols)
    fig, axs = plt.subplots(
        nrows, ncols, figsize=(8*ncols, 8*nrows)
    )
    axs = axs.flatten()
    for ax, plot_data in zip(axs, grid_plot_data):
        grid_plot_fn(ax, plot_data)
    return fig, axs

In [ ]:
def pretty_format_duration(duration: float) -> str:
    out_seconds = duration % 60
    dur_minutes = int(duration / 60)
    if dur_minutes == 0:
        return f"{out_seconds:.1f} s"
    out_minutes = dur_minutes % 60
    dur_hours = int(dur_minutes / 60)
    if dur_hours == 0:
        return f"{out_minutes} m {out_seconds:.1f} s"
    else:
        return f"{dur_hours} h {out_minutes} m {out_seconds:.1f} s"

## Data Loading

### Loading Params

In [ ]:
# experiment_ids = ["TB-26"]
experiment_ids = ["ID-418"]

In [ ]:
base_path = Path("unfolding_test")
test_file = base_path / "sigma_50keV_FWHM.csv.npy"

In [ ]:
# R_folder_name = "response_matrix_geant_point"
# R_folder_name = "response_matrix_geant_gaussian"
# R_folder_name = "response_matrix_geant_Tbird_WithSig"
R_folder_name = "response_matrix_R4_mono"
# R_folder_name = "response_matrix_11MeV"

In [ ]:
energy = "2.480_MeV"
unc_energy_filename = f"neutron_{energy}.csv.npy"
unc_energy_data = {"filepath": Path(R_folder_name) / unc_energy_filename}

In [ ]:
# more here?
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}

In [ ]:
# calib_input = get_input_with_default(
#     "Do you want to use new calibration? [y/n, or press Enter for yes]",
#     "y",
#     str
# )
calib_input = "y"

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column: EnergyColumn = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = ExperimentDataKey.NEW_CALIBRATION if is_new_calibration else ExperimentDataKey.CAEN_CALIBRATION

In [ ]:
# detector_code = helpers.get_input_required(
#     """\
# Which detector was used?
# 1: Original detector (detector 1)
# 2: New detector (detector 2)
# """,
#     [Detector.ZERO, Detector.ONE],
#     lambda x: Detector(int(x)-1)
# )
detector_code = proc.Detector.ZERO

In [ ]:
default_fit_input = 2  # changed to peak finder mode, approved by Fatima 2024-07-18
# fit_input = helpers.get_input_with_default(
#     """\
# Which bimodal fit type do you want to use?
# 1: Bounds based
# 2: Peak finder based (default)
# Press Enter for default
# """,
#     default_fit_input,
#     int
# )
fit_input = 2

fit_styles: dict[int, proc_types.SliceFitStyle] = {
    1: "bounds",
    2: "peak_finder"
}
fit_style = fit_styles.get(fit_input, fit_styles[default_fit_input])

In [ ]:
strategy_factory = proc.NeutronStrategyFactory()
# settings = get_nasa_generation_settings(calib_key)
window_offset = 0.2
sigma = 5
lower_energy_bound = 0
recalc_lower_bound = False
settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
factory_fn = make_strategy_factory_fn(
    strategy_factory, "nasa", False, settings)
experiment_neutron_data = make_strategy_for_experiments(
    experiment_neutron_data, factory_fn)

In [ ]:
# bin_length = helpers.get_input_with_default(
#     "Enter bin length (in seconds), or press Enter for default (300 s)",
#     300,
#     int
# )
bin_length = 30
bin_string = f"{bin_length}S"

In [ ]:
# bins_min = helpers.get_input_with_default(
#     "Enter minimum light output (in MeVee), or press Enter for default (0 MeVee)",
#     0,
#     float
# )
# bins_max = helpers.get_input_with_default(
#     "Enter maximum light output (in MeVee), or press Enter for default (1.2 MeVee)",
#     6,
#     float
# )
# bins_width = helpers.get_input_with_default(
#     "Enter light output bin width (in MeVee), or press Enter for default (0.02 MeVee)",
#     0.02,
#     float
# )
l_bins_min = 0
l_bins_max = 6
l_bins_width = 0.02

L_bins = np.arange(l_bins_min, l_bins_max + l_bins_width, l_bins_width).tolist()

In [ ]:
analysis_timestamp = datetime.now().strftime("%Y-%m%b-%d-%H-%M-%S")
overall_settings = {
    'calibration_type': repr(calib_key),
    'fitting_style': fit_style,
    'window_settings': repr(settings),
    'bin_length': bin_length
}

In [ ]:
analysis_timestamp

### Loading and Initial Processing

In [ ]:
figure_data = {k: {} for k in ["a", "b1", "b2"]}

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    # exp_data[ExperimentDataKey.UNCLASSIFIED] = load_parquet_psd(exp_id)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = load.load_caen_csvs(exp_id, raw=True)

In [ ]:
R = load.load_neutron_response_matrix(
    Path(R_folder_name),
    min_L=l_bins_min,
    max_L=l_bins_max,
    L_bin_widths=l_bins_width
)

In [ ]:
bins = np.arange(l_bins_min, l_bins_max + l_bins_width, l_bins_width)
np_Ls = (bins[1:] + bins[:-1]) / 2

In [ ]:
L_array = np.load(test_file)

np_cps, *_ = np.histogram(L_array, bins=bins)
np_cps = np_cps.reshape(-1, 1)
ddN = proc.NDHistogram(np_cps, [np_Ls, np.ones(1)])

In [ ]:
mov_avg_window = 7
polyorder = 3

filepath = unc_energy_data["filepath"]
L_array = np.load(filepath)
np_cps, *_ = np.histogram(L_array, bins=bins)
np_cps_savgol = savgol_filter(np_cps, window_length=mov_avg_window, polyorder=polyorder)
np_cps_savgol = np_cps_savgol.reshape(-1, 1)
base_N = proc.NDHistogram(np_cps_savgol, [np_Ls, np.ones(1)])
unc_energy_data["base_N"] = base_N

In [ ]:
# Express timetags in hours elapsed
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = load.calculate_timetag_hours(unclassified_df)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Recalibrate energy
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = proc.recalibrate(unclassified_df, detector_code)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Generate histogram

start_scan_idx = 0
end_scan_idx = 420
energy_width = 5e-3
overall_settings['scan_idx'] = f"({start_scan_idx}, {end_scan_idx})"
overall_settings['energy_width'] = energy_width

for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    Z, xe, ye = proc.get_psd_energy_histogram(
        psd_report,
        calibrated_energy_column,
        energy_width=energy_width
    )
    exp_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
    exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
    exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye
    exp_data[ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z))

In [ ]:
# get fit dataframe (not needed if loading, but do anyway to keep process consistent)
stop_here = False

for exp_id, exp_data in experiment_neutron_data.items():
    Z = exp_data[ExperimentDataKey.PSD_HISTOGRAM]
    xe = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    end_scan_idx = exp_data[ExperimentDataKey.END_SCAN_IDX]

    if fit_style == "bounds":
        # Default
        default_bounds: proc.BimodalBounds = (
            proc.BimodalParams(0.1, 0.01, 1, 0.25, 0.01, 0),
            proc.BimodalParams(0.2, 0.1, Z.max(), 0.38, 0.04, 4000)
        )

        bounds_a: proc.BimodalBounds = (
            proc.BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
            proc.BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.04, 4000)
        )

        bounds_b: proc.BimodalBounds = (
            proc.BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
            proc.BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.03, 4000)
        )

        # Ranged Example
        bounds = [
            ((0, 60), bounds_a),
        ]
    else:
        default_bounds = None
        bounds = None

    df, df_err = proc.scan_histogram_slices(
        Z,
        xe,
        ye,
        fit_style=fit_style,
        default_bounds=default_bounds,
        bounds=bounds,
        start_idx=start_scan_idx,
        end_idx=end_scan_idx
    )
    df, bad_slice_indexes = proc.find_failed_slices(df, exp_id, nan_total_threshold=10)

    if bad_slice_indexes is not None:
        exp_data[ExperimentDataKey.VALID_SLICE_FITS] = df
        exp_data[ExperimentDataKey.BAD_SLICE_INDEXES] = bad_slice_indexes
        stop_here = True
    else:
        # exp_data['fom_results'] = df
        exp_data[ExperimentDataKey.FOM_RESULTS] = df

if stop_here:
    helpers.stop()

In [ ]:
# get borders from strategy
for exp_id, exp_data in experiment_neutron_data.items():
    if ExperimentDataKey.FOM_RESULTS not in exp_data:
        print(f"No good fit data on Experiment {exp_id}")
        continue

    fom_results = exp_data[ExperimentDataKey.FOM_RESULTS]
    strategy = exp_data[ExperimentDataKey.BORDER_STRATEGY]

    strategy.set_slice_fit_dataframe(fom_results)
    borders = strategy.get_neutron_window()

    exp_data[ExperimentDataKey.BORDERS] = borders

In [ ]:
# classify neutrons
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED].copy()
    borders = exp_data[ExperimentDataKey.BORDERS]

    psd_report = proc.classify(
        psd_report,
        calibrated_energy_column,
        borders,
        DetectorDataframeColumn.NEW_N_CLASS
    )

    exp_data[ExperimentDataKey.PSD_REPORT] = psd_report

In [ ]:
# Get experiment start time
for exp_name, data_dict in experiment_neutron_data.items():
    exp_root = get_exp_root(exp_name)
    with open(exp_root / 'exp_info.toml') as exp_info:
        exp_start_line = [line for line in exp_info if "exp_start" in line][0]
    exp_start_text = exp_start_line.replace("exp_start = ", "").strip()
    exp_start = datetime.fromisoformat(exp_start_text).astimezone(timezone.utc)
    data_dict[ExperimentDataKey.START_TIME] = exp_start

In [ ]:
# Get timetag as clock time
for exp_name, data_dict in experiment_neutron_data.items():
    psd_report = data_dict[ExperimentDataKey.PSD_REPORT]
    exp_start = data_dict[ExperimentDataKey.START_TIME]

    psd_report = load.calculate_event_time(psd_report, exp_start)

    data_dict[ExperimentDataKey.PSD_REPORT] = psd_report

In [ ]:
# Separate neutron and gamma events
for exp_name, data_dict in experiment_neutron_data.items():
    psd_report = data_dict[ExperimentDataKey.PSD_REPORT]

    n_classify_col_name = DetectorDataframeColumn.NEW_N_CLASS.value
    neutrons_only = psd_report.query(n_classify_col_name).copy()
    gamma_only = psd_report.query(f"~{n_classify_col_name}").copy()
    data_dict[ExperimentDataKey.NEUTRONS_ONLY] = neutrons_only
    data_dict[ExperimentDataKey.GAMMA_ONLY] = gamma_only

In [ ]:
# Get total experiment time
for exp_name, data_dict in experiment_neutron_data.items():
    neutrons_only = data_dict[ExperimentDataKey.NEUTRONS_ONLY]
    max_timetag = neutrons_only["TIMETAG"].max()
    data_dict["max_seconds"] = max_timetag * 1E-12

## Unfold Neutron Spectrum

In [ ]:
exp_id = "ID-418"
exp_data = experiment_neutron_data[exp_id]

### PHD

In [ ]:
# make PSD histogram
neutrons_only = exp_data[ExperimentDataKey.NEUTRONS_ONLY]
max_seconds = exp_data["max_seconds"]
neutron_energies = neutrons_only[calibrated_energy_column.value]
energy_bins = np.arange(start=l_bins_min, stop=l_bins_max + l_bins_width, step=l_bins_width)

Z_n, *_ = np.histogram(neutron_energies, bins=energy_bins)
exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION] = {
    "neutron": {"standard": Z_n, "standard_edges": energy_bins}
}

In [ ]:
phd_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["neutron"]
counts = phd_data["standard"]
counts[2] = 0
counts[143] = 0
phd_data["standard"] = counts


### Normalization

In [ ]:
# Normalization
# By E slice sum
R_mids0, R_mids1 = R.midpoints
R_sum = R.counts.sum(axis=0, keepdims=True)
# R_norm = R.counts / R_sum
# print(R_norm.sum(axis=0))
R_norm = R.counts
R = proc.NDHistogram(R_norm, R.midpoints)

In [ ]:
# Normalization
phd_neutron_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION][
    "neutron"]
Z_n = phd_neutron_data["standard"]
Z_n = Z_n.reshape(-1, 1)

counts_sum = Z_n.sum()
# Z_norm = Z_n / counts_sum
Z_norm = Z_n

# print(Z_norm.sum())
# print(Z_norm.shape)
phd_neutron_data["normalized"] = Z_norm

In [ ]:
R_L_mids, R_E_mids = R.midpoints
phd_neutron_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION][
    "neutron"]
PHD = phd_neutron_data["normalized"]
PHD_bins = phd_neutron_data["standard_edges"]
PHD_mids = helpers.get_midpoints_from_bins(PHD_bins)
if (
    PHD_mids.shape[0] != R_L_mids.shape[0]
    or not all(np.isclose(PHD_mids, R_L_mids))
):
    raise ValueError("Bin mismatch!")
reduced_E_mids = np.array([R_E_mids.mean()])
phd_neutron_data["standard_histogram"] = proc.NDHistogram(
    PHD, [R_L_mids, reduced_E_mids]
)

In [ ]:
# print(type(ddN))
ddN_counts = ddN.counts
counts_sum = ddN_counts.sum()
# ddN_normed = ddN_counts / counts_sum
ddN_normed = ddN_counts

# print(ddN_normed.sum())
# print(ddN_normed.shape)
ddN = proc.NDHistogram(ddN_normed, ddN.midpoints)

In [ ]:
# tol = helpers.get_input_with_default(
#     "Enter unfolding tolerance, or press Enter for default (0.01)",
#     0.01,
#     float
# )
# tol = 1e-12
tol = 1e-9

In [ ]:
print(R.shape)

In [ ]:
N = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["neutron"]["standard_histogram"]
phi, unfold_info = proc.unfold_spectrum(R, N, full_info=True, tolerance=tol, L_cut=0.05)
exp_data["neutron_spectrum"] = phi
exp_data["unfolding_info"] = unfold_info

In [ ]:
dd_phi, *_ = proc.unfold_spectrum(
    R,
    ddN,
    L_cut=0.05,
    # tolerance=0.0000001,
    tolerance=tol,
    max_iterations=1000
)

In [ ]:
# phi = exp_data["neutron_spectrum"]
# phi_flat = phi.counts.reshape(-1)
# print(np.nansum(phi_flat))
# phi_norm = np.linalg.norm(phi_flat[~np.isnan(phi_flat)])
# phi_normed = phi_flat/phi_norm
# # exp_data["spectrum_flat"] = phi_flat
# # exp_data["spectrum_normalized"] = phi_normed
# phi_normed = phi_normed.reshape(1, -1)
# exp_data["spectrum_normalized"] = proc.NDHistogram(phi_normed, phi.midpoints)

# dd_phi_flat = dd_phi.counts.reshape(-1)
# print(np.nansum(dd_phi_flat))
# dd_norm = np.linalg.norm(dd_phi_flat[~np.isnan(dd_phi_flat)])
# dd_phi_normed = dd_phi_flat/dd_norm
# dd_phi_normed = dd_phi_normed.reshape(1, -1)
# dd_phi_normed = proc.NDHistogram(dd_phi_normed, dd_phi.midpoints)

In [ ]:
# dd_phi_norm_sum = np.nansum(dd_phi_normed.counts)
# print(dd_phi_norm_sum)

# time_dict = {
#     "ID-418": 7860,
#     "ID-422": 7620,
#     "ID-423": 7620
# }

# N = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["neutron"]["standard_histogram"]
# phi = exp_data["neutron_spectrum"]
# phi_normed = exp_data["spectrum_normalized"]
# exp_time = time_dict[exp_id]

# Nsum = N.counts.sum()
# phi_norm_sum = np.nansum(phi_normed.counts)
# print(phi_norm_sum)
# corr_factor = (Nsum / phi_norm_sum) / exp_time
# phi_corr = phi_normed * corr_factor
# print(np.nansum(phi_corr))
# exp_data["spectrum_corrected"] = phi_corr

# dd_corr_factor = (Nsum / dd_phi_norm_sum) / exp_time
# dd_phi_corr = dd_phi_normed * dd_corr_factor
# print(np.nansum(dd_phi_corr))
# exp_data["dd_spectrum_corrected"] = dd_phi_corr

### Uncertainty Estimation by Perturbation

In [ ]:
# Basic unfolding
base_N = unc_energy_data["base_N"]
phi, unfold_info = proc.unfold_spectrum(
    R,
    base_N,
    L_cut=0.05,
    tolerance=tol,
    max_iterations=1000,
    full_info=True
)
unc_energy_data["no_shift"] = {
    "phi": phi,
    "unfold_info": unfold_info
}

In [ ]:
base_N = unc_energy_data["base_N"]
N_counts = base_N.counts
sigma_counts = np.sqrt(N_counts)
sigma = proc.NDHistogram(sigma_counts, N.midpoints)

In [ ]:
# Up shift
N_yerr_plus = proc.NDHistogram(N_counts + 3 * sigma_counts, N.midpoints)
phi_yerr_plus, _ = proc.unfold_spectrum(
    R,
    N_yerr_plus,
    L_cut=0.05,
    tolerance=tol,
    max_iterations=1000,
    sigma=sigma
)
unc_energy_data["upshift"] = {
    "N": N_yerr_plus,
    "phi": phi_yerr_plus
}

In [ ]:
# Down shift
N_yerr_minus = proc.NDHistogram(N_counts - 3 * sigma_counts, N.midpoints)
phi_yerr_minus, _ = proc.unfold_spectrum(
    R,
    N_yerr_minus,
    L_cut=0.05,
    tolerance=tol,
    max_iterations=1000,
    sigma=sigma
)
unc_energy_data["upshift"] = {
    "N": N_yerr_minus,
    "phi": phi_yerr_minus
}

In [ ]:
sigma = 0.050  # MeVee

In [ ]:
# Left shift
# lshift_mids = base_N.midpoints[0] - 3 * 

In [ ]:
# Right shift

### Unfolding Errors

#### Horizontal

In [ ]:
base_xerr = [(1.798, (0.31000000000000005, 0.6200000000000001)),
 (1.86, (0.31000000000000005, 0.558)),
 (1.922, (0.3719999999999999, 0.558)),
 (1.984, (0.3719999999999999, 0.496)),
 (2.046, (0.3719999999999999, 0.496)),
 (2.108, (0.3720000000000001, 0.496)),
 (2.17, (0.43399999999999994, 0.43400000000000016)),
 (2.232, (0.3720000000000001, 0.3719999999999999)),
 (2.294, (0.3720000000000001, 0.3719999999999999)),
 (2.418, (0.43400000000000016, 0.31000000000000005)),
 (2.48, (0.496, 0.31000000000000005)),
 (2.604, (0.3719999999999999, 0.4339999999999997)),
 (2.666, (0.4339999999999997, 0.3719999999999999)),
 (2.79, (0.496, 0.3719999999999999)),
 (2.852, (0.3719999999999999, 0.43400000000000016)),
 (2.914, (0.43400000000000016, 0.3719999999999999)),
 (2.976, (0.496, 0.18599999999999994)),
 (3.038, (0.496, 0.1860000000000004)),
 (3.1, (0.5580000000000003, 0.18599999999999994)),
 (3.162, (0.496, 0.18599999999999994)),
 (3.224, (0.5580000000000003, 0.18599999999999994)),
 (3.286, (0.5579999999999998, 0.18599999999999994))]

In [ ]:
phi = exp_data["neutron_spectrum"]
phi_mids = phi.midpoints[1]

ddphi_mids = dd_phi.midpoints[1]

In [ ]:
phi_xerr = []
for mid in phi_mids:
    isclose = [np.isclose(x, mid) for x, _ in base_xerr]
    true_idx = [i for i, x in enumerate(isclose) if x]
    if len(true_idx) == 0:
        phi_xerr.append((np.nan, np.nan))
        continue
    idx = true_idx[0]
    _, errorbar = base_xerr[idx]
    phi_xerr.append(errorbar)
phi_xerr = list(zip(*phi_xerr))
exp_data["neutron_spectrum_xerr"] = phi_xerr

In [ ]:
ddphi_xerr = []
for mid in ddphi_mids:
    isclose = [np.isclose(x, mid) for x, _ in base_xerr]
    true_idx = [i for i, x in enumerate(isclose) if x]
    if len(true_idx) == 0:
        ddphi_xerr.append((np.nan, np.nan))
        continue
    idx = true_idx[0]
    _, errorbar = base_xerr[idx]
    ddphi_xerr.append(errorbar)
ddphi_xerr = list(zip(*ddphi_xerr))

#### Vertical

In [ ]:
N = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["neutron"]["standard_histogram"]
phi = exp_data["neutron_spectrum"]
phi_xerr = exp_data["neutron_spectrum_xerr"]

N_sigma_counts = np.sqrt(N.counts)
N_upshift = proc.NDHistogram(N.counts + 3 * N_sigma_counts, N.midpoints)
N_downshift = proc.NDHistogram(N.counts - 3 * N_sigma_counts, N.midpoints)

phi_upshift, _ = proc.unfold_spectrum(
    R,
    N_upshift,
    L_cut=0.05,
    tolerance=tol
)
phi_downshift, _ = proc.unfold_spectrum(
    R,
    N_downshift,
    L_cut=0.05,
    tolerance=tol
)

phi_flat = phi.counts.reshape(-1)
phi_up_flat = phi_upshift.counts.reshape(-1)
phi_down_flat = phi_downshift.counts.reshape(-1)

up_gt_phi = (phi_up_flat >= phi_flat) | (np.isnan(phi_flat))
down_lt_phi = (phi_down_flat <= phi_flat) | (np.isnan(phi_flat))
phi_yerr_larger = phi_up_flat.copy()
phi_yerr_larger[~up_gt_phi] = phi_down_flat[~up_gt_phi]
phi_yerr_smaller = phi_down_flat.copy()
phi_yerr_smaller[~down_lt_phi] = phi_up_flat[~down_lt_phi]

phi_yerr_up = np.nan_to_num(phi_yerr_larger - phi_flat, nan=np.nan)
phi_yerr_down = np.nan_to_num(phi_flat - phi_yerr_smaller, nan=np.nan)
phi_yerr_up[phi_yerr_up < 0] = 0
phi_yerr_down[phi_yerr_down < 0] = 0

phi_xerr_left, _ = phi_xerr
phi_yerr_up[np.isnan(phi_xerr_left)] = np.nan
phi_yerr_down[np.isnan(phi_xerr_left)] = np.nan

exp_data["neutron_spectrum_yerr"] = [phi_yerr_down, phi_yerr_up]
# phi, unfold_info = proc.unfold_spectrum(R, N, full_info=True, tolerance=tol, L_cut=0.05)
# exp_data["neutron_spectrum"] = phi
# exp_data["unfolding_info"] = unfold_info

In [ ]:
ddN_sigma_counts = np.sqrt(ddN.counts)
ddN_upshift = proc.NDHistogram(ddN.counts + 3 * ddN_sigma_counts, ddN.midpoints)
ddN_downshift = proc.NDHistogram(ddN.counts - 3 * ddN_sigma_counts, ddN.midpoints)

ddphi_upshift, _ = proc.unfold_spectrum(
    R,
    ddN_upshift,
    L_cut=0.05,
    tolerance=tol
)
ddphi_downshift, _ = proc.unfold_spectrum(
    R,
    ddN_downshift,
    L_cut=0.05,
    tolerance=tol
)

ddphi_flat = dd_phi.counts.reshape(-1)
ddphi_up_flat = ddphi_upshift.counts.reshape(-1)
ddphi_down_flat = ddphi_downshift.counts.reshape(-1)

up_gt_ddphi = (ddphi_up_flat >= ddphi_flat) | (np.isnan(ddphi_flat))
down_lt_ddphi = (ddphi_down_flat <= ddphi_flat) | (np.isnan(ddphi_flat))
ddphi_yerr_larger = ddphi_up_flat.copy()
ddphi_yerr_larger[~up_gt_ddphi] = ddphi_down_flat[~up_gt_ddphi]
ddphi_yerr_smaller = ddphi_down_flat.copy()
ddphi_yerr_smaller[~down_lt_ddphi] = ddphi_up_flat[~down_lt_ddphi]

ddphi_yerr_up = np.nan_to_num(ddphi_yerr_larger - ddphi_flat, nan=np.nan)
ddphi_yerr_down = np.nan_to_num(ddphi_flat - ddphi_yerr_smaller, nan=np.nan)
ddphi_yerr_up[ddphi_yerr_up < 0] = 0
ddphi_yerr_down[ddphi_yerr_down < 0] = 0

ddphi_xerr_left, _ = ddphi_xerr
ddphi_yerr_up[np.isnan(ddphi_xerr_left)] = np.nan
ddphi_yerr_down[np.isnan(ddphi_xerr_left)] = np.nan

ddphi_yerr = (ddphi_yerr_down, ddphi_yerr_up)

### Subplot Processing

#### Figure 21a

In [ ]:
exp_id = "ID-418"
plot_data = figure_data["a"]

In [ ]:
exp_data = experiment_neutron_data[exp_id]
phi = exp_data["neutron_spectrum"]
phi_yerr = exp_data["neutron_spectrum_yerr"]
phi_xerr = exp_data["neutron_spectrum_xerr"]
plot_data["phi"] = {
    "counts": phi.counts.reshape(-1),
    "mids": phi.midpoints[1],
    "xerr": phi_xerr,
    "yerr": phi_yerr,
}
plot_data["dd"] = {
    "counts": dd_phi.counts.reshape(-1),
    "mids": dd_phi.midpoints[1],
    "xerr": ddphi_xerr,
    "yerr": ddphi_yerr
}

## Plotting

### Plot Style Constants

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

In [ ]:
bg_blue = "#4c94ff"
bg_red = "#f54336"
bg_grey = "#9e9e9e"
bg_bluegrey = "#8a9fb8"
bg_green = "#21d894"
transparent = "#00000000"

### Plot Functions

In [ ]:
def plot_figure_21a(
    ax: mpl.axes.Axes,
    dd_scaling: float
):
    plot_data = figure_data["a"]
    phi_data = plot_data["phi"]
    dd_data = plot_data["dd"]
    phi_counts = phi_data["counts"]
    phi_mids = phi_data["mids"]
    phi_xerr = phi_data["xerr"]
    phi_yerr = phi_data["yerr"]
    dd_counts = dd_data["counts"] * dd_scaling
    dd_mids = dd_data["mids"]
    dd_xerr = dd_data["xerr"]
    dd_yerr = dd_data["yerr"]
    dd_yerr = [dd_yerr_side * dd_scaling for dd_yerr_side in dd_yerr]
    
    plot_params = {
        "lw": 3,
        "marker": "o",
        "ms": 5,
        "elinewidth": 2
    }
    ax.errorbar(
        phi_mids, phi_counts,
        xerr=phi_xerr,
        yerr=phi_yerr,
        **plot_params,
        color=bg_blue
    )
    ax.errorbar(
        dd_mids, dd_counts,
        xerr=dd_xerr,
        yerr=dd_yerr,
        **plot_params,
        color=bg_bluegrey)

    ax.set_xlabel("E (MeV)", fontsize=fontsize)
    ax.set_ylabel("Normalized counts", fontsize=fontsize)
    ax.tick_params(labelsize=fontsize)
    ax.set_xlim(1.5, 3.5)

# dd_scaling = 1
# for exp_id, exp_data in experiment_neutron_data.items():
#     print(exp_id)
#     # phi = exp_data["neutron_spectrum"]
#     # phi_flat = exp_data["spectrum_flat"]
#     fig, ax = plt.subplots(
#         # figsize=figsize,
#         dpi=300
#     )
#     ax.plot(phi.midpoints[1], phi_flat, marker="o", markersize=3, label="Experimental spectrum")
#     ax.plot(dd_phi_mids, dd_phi_flat * dd_scaling, marker="o", markersize=3, label="DD simulation")
#     ax.set_xlabel("E (MeV)", fontsize=fontsize)
#     ax.set_ylabel("Normalized counts", fontsize=fontsize)
#     # ax.set_yscale("log")
#     ax.legend()
#     ax.tick_params(labelsize=fontsize)
#     plt.show()

In [ ]:
fig, ax = plt.subplots(layout="constrained")
plot_figure_21a(ax, 30)